# ONT nhmmer Spidroin Full-Length Screening

Screen ONT isoform transcripts directly with nucleotide HMMER (nhmmer) terminal-domain models from Schoneberg et al. 2025, filter transcripts containing both NTD and CTD domains, and determine the final CDS sequence bounded by start/stop codons.

Intermediate files are written to `data/interim/{TASK_NAME}/`; final tables and FASTA files are written to `data/processed/{TASK_NAME}/`.

## Configuration

Import packages and define run-specific paths. `TASK_NAME` includes a timestamp suffix so each run is isolated.

In [ ]:
from datetime import datetime

import polars as pl

from spider_silkome_module import (
    INTERIM_DATA_DIR,
    PROCESSED_DATA_DIR,
    PROJ_ROOT,
    REFERENCES_DIR,
    run_cmd,
)

In [ ]:
TASK_NAME = f"ont_nhmmer_spidroin_20260623-2"

isoform_dir = PROJ_ROOT / "workflow" / "ONT_RNA_align" / "results" / "isoforms"
hmm_dir = REFERENCES_DIR / "2025_Schoneberg_data" / "hmmer_nucl_profile_trimmed"

interim_dir = INTERIM_DATA_DIR / TASK_NAME
processed_dir = PROCESSED_DATA_DIR / TASK_NAME

THREADS = 70
EVALUE = 1e-10
COVERAGE = 0.90
FORCE = False

interim_dir, processed_dir

## Step 1: Preflight

Check that required executables (nhmmer, hmmpress) and input directories are available.

In [ ]:
preflight_tsv = interim_dir / "preflight.tsv"

run_cmd(
    f"pixi run python -m spider_silkome_module.ont_nhmmer_spidroin preflight \
        --isoform-dir {isoform_dir} \
        --hmm-dir {hmm_dir} \
        --output {preflight_tsv}",
    outputs=[preflight_tsv],
    force=True,
)

## Step 2: Discover Species

Build a manifest of ONT transcript FASTA files (one per species).

In [ ]:
manifest_tsv = interim_dir / "species_manifest.tsv"

run_cmd(
    f"pixi run python -m spider_silkome_module.ont_nhmmer_spidroin discover \
        --isoform-dir {isoform_dir} \
        --output {manifest_tsv}",
    outputs=[manifest_tsv],
    force=FORCE,
)

manifest = pl.read_csv(manifest_tsv, separator="\t")
manifest

## Step 3: Run Full Workflow

Run nhmmer search on transcript FASTAs against the combined nucleotide HMM profiles, parse and filter hits, classify transcripts by NTD/CTD presence, and determine CDS sequences for full-length candidates.

In [ ]:
target_tsv = processed_dir / "target_classification.tsv"

run_cmd(
    f"pixi run python -m spider_silkome_module.ont_nhmmer_spidroin run \
        --isoform-dir {isoform_dir} \
        --hmm-dir {hmm_dir} \
        --interim-dir {interim_dir} \
        --processed-dir {processed_dir} \
        --threads {THREADS} \
        --evalue {EVALUE} \
        --coverage {COVERAGE}",
    outputs=[target_tsv],
    force=True,
)

## Results

Inspect final full-length candidates, CDS FASTA, pivot counts, and validation checks.

In [ ]:
target_classification = pl.read_csv(processed_dir / "target_classification.tsv", separator="\t")
full_length = pl.read_csv(processed_dir / "full_length_spidroins.tsv", separator="\t")
partial = pl.read_csv(processed_dir / "partial_spidroins.tsv", separator="\t")
pivot = pl.read_csv(processed_dir / "spidroin_pivot_table.tsv", separator="\t")
validation = pl.read_csv(processed_dir / "validation_summary.tsv", separator="\t")

print(f"target classifications: {target_classification.shape}")
print(f"full-length candidates: {full_length.shape}")
print(f"partial/non-full candidates: {partial.shape}")

display(validation)
display(pivot)
full_length.head()

## Output Files

List all generated output paths for downstream use.

In [ ]:
output_files = {
    "target_classification": processed_dir / "target_classification.tsv",
    "full_length_spidroins": processed_dir / "full_length_spidroins.tsv",
    "partial_spidroins": processed_dir / "partial_spidroins.tsv",
    "full_length_cds_fasta": processed_dir / "full_length_spidroins_cds.fasta",
    "full_length_protein_fasta": processed_dir / "full_length_spidroins_protein.fasta",
    "full_length_pair_audit": processed_dir / "full_length_pair_audit.tsv",
    "spidroin_pivot_table": processed_dir / "spidroin_pivot_table.tsv",
    "validation_summary": processed_dir / "validation_summary.tsv",
}
output_files